# Lab 0 — Getting Set Up

**Computer Vision for CS &amp; AI** &nbsp;|&nbsp; two hours &nbsp;|&nbsp; do this **before** Lecture 1 and Lab 1

---

### Read this before you run anything

**Click `File → Save a copy in Drive` right now.** If you skip this, nothing
you do today is saved. Rename your copy `CV_Lab00_yourname`.

This lab is not marked for correctness. It is marked as done or not done, and
it is compulsory — its whole purpose is that you never spend Lab 1 fighting
your environment.

Work through the sections in order. The last cell tells you whether you are
ready.


---
## 1 · Check the toolchain &nbsp;&nbsp;<small>(0:15–0:30)</small>

Nothing needs installing — Colab ships with everything this course uses.
This cell proves it. If an assertion fails, that is a real finding: report
it rather than working around it.


In [ ]:
import sys, os, platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import skimage
import pandas as pd
from PIL import Image
from skimage import data, color, io

print('python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
print('matplotlib ', matplotlib.__version__)
print('skimage    ', skimage.__version__)
print('pandas     ', pd.__version__)
print('platform   ', platform.platform())

for name, mod in [('numpy', np), ('skimage', skimage), ('pandas', pd)]:
    assert mod is not None, f'{name} failed to import'
print('\nToolchain OK — nothing to install.')


---
## 2 · Load a dataset, three ways &nbsp;&nbsp;<small>(0:30–0:55)</small>

### 2a. From the bundled samples — the way almost every lab works

These photographs ship inside `scikit-image`. Nothing downloads, so a slow
connection or a dead dataset link can never stop you.


In [ ]:
img = data.chelsea()          # bundled sample; follow scikit-image attribution terms

# The four facts. Print these before anything else, every single time.
print('shape :', img.shape)
print('dtype :', img.dtype)
print('min   :', img.min())
print('max   :', img.max())

plt.imshow(img); plt.axis('off'); plt.title('data.chelsea()'); plt.show()


**Q2.1** The shape is three numbers. What does each one mean?

*Your answer:*

### 2b. From a file you upload

Run the cell, pick any photograph from your laptop. Skip it if you have none
to hand — but you will need this in Lab 11, so it is worth doing once now.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # opens a file picker
    if uploaded:
        name = list(uploaded)[0]
        mine = io.imread(name)
        print(name, mine.shape, mine.dtype, mine.min(), mine.max())
        plt.imshow(mine); plt.axis('off'); plt.show()
except ImportError:
    print('Not running in Colab — skip this cell.')


### 2c. From a URL

One line, and the same four facts afterwards.


In [ ]:
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/'\
      '4/47/PNG_transparency_demonstration_1.png/280px-'\
      'PNG_transparency_demonstration_1.png'
try:
    remote = io.imread(url)
    print('shape', remote.shape, '| dtype', remote.dtype,
          '| min', remote.min(), '| max', remote.max())
except Exception as e:
    print('Could not fetch (this is survivable — the course does not depend on it):', e)


**Q2.2** If that last image loaded, its shape may have **four** channels,
not three. What is the fourth, and what would it do to code that assumes three?

*Your answer:*


---
## 3 · Mount your Drive and build the course folder &nbsp;&nbsp;<small>(0:55–1:15)</small>

You will approve a permission prompt. Watch for it — dismissing it is the
most common problem in this lab.

If Drive is blocked at your institution, change `DRIVE` to
`'/content/cv_course'` and carry on. You lose persistence between sessions,
not the lab.


In [ ]:
DRIVE = '/content/drive/MyDrive/cv_course'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, Exception) as e:
    DRIVE = '/content/cv_course'
    print('Drive unavailable, falling back to', DRIVE, '-', e)

for sub in ('data', 'outputs', 'notebooks'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)

print('\nCourse folder:', DRIVE)
for entry in sorted(os.listdir(DRIVE)):
    print('  ', entry + '/')


---
## 4 · Write something, then prove you wrote it &nbsp;&nbsp;<small>(1:15–1:40)</small>

Saving without verifying is how people lose a week's work. Every save in
this course is followed by a reload and an assertion.

### 4a. Watch the range change

`rgb2gray` returns **floats in 0–1**, not integers in 0–255. This catches
almost everyone once. Let it catch you here, where it costs nothing.


In [ ]:
gray = color.rgb2gray(img)
print('grayscale dtype:', gray.dtype, '| min', round(gray.min(), 3),
      '| max', round(gray.max(), 3))
print('original  dtype:', img.dtype, ' | min', img.min(), '| max', img.max())

# Convert to 8-bit before saving, or the PNG comes out black.
gray_u8 = (gray * 255).astype(np.uint8)
print('converted dtype:', gray_u8.dtype, '| min', gray_u8.min(),
      '| max', gray_u8.max())


### 4b. Save, reload, assert


In [ ]:
PNG = f'{DRIVE}/outputs/lab00_gray.png'

io.imsave(PNG, gray_u8)
print('wrote', PNG)

reloaded = io.imread(PNG)
print('read back:', reloaded.shape, reloaded.dtype)

assert reloaded.shape == gray_u8.shape, 'shape changed on the round trip'
print('Round trip verified.')


### 4c. Store measurements, and read them back

A result you cannot regenerate is not a result. From today, everything you
report in this course is a file produced by code you can run again.


In [ ]:
CSV = f'{DRIVE}/outputs/lab00_results.csv'

results = pd.DataFrame({
    'measurement': ['height', 'width', 'channels', 'mean_intensity',
                    'std_intensity'],
    'value': [img.shape[0], img.shape[1], img.shape[2],
              round(float(img.mean()), 3), round(float(img.std()), 3)],
})
results.to_csv(CSV, index=False)
print('wrote', CSV)

back = pd.read_csv(CSV)
print(back.to_string(index=False))
assert len(back) == 5, 'expected five rows'
print('\nResults round trip verified.')


**Q4.1** Why does this notebook reload every file it writes, rather than
trusting that the save worked?

*Your answer:*


---
## 5 · Make it reproducible &nbsp;&nbsp;<small>(1:40–1:52)</small>

Run the cell, note the numbers. Then `Runtime → Restart runtime`, run it
again, and confirm they are identical.


In [ ]:
np.random.seed(0)
print('seeded  :', np.random.rand(3).round(4))

rng = np.random.default_rng(0)      # the modern form, used in this course
print('generator:', rng.random(3).round(4))


**Now do the thing that matters most.**

`Runtime → Restart and run all`. The notebook must complete top to bottom
with no errors. A notebook that only works when cells are run out of order
is not finished — and this is the single most common reason a submitted lab
scores zero.


---
## 6 · Self-test &nbsp;&nbsp;<small>(1:52–2:00)</small>

Run this last. Fix anything that says FAIL.


In [ ]:
checks = {}

def check(name, fn):
    try:
        checks[name] = bool(fn())
    except Exception as e:
        checks[name] = False
        print(f'   ({name}: {type(e).__name__} - {e})')

check('toolchain imports',        lambda: np and skimage and pd)
check('dataset loads',            lambda: data.chelsea().shape == (300, 451, 3))
check('four facts printed',       lambda: img.dtype == np.uint8 and img.max() <= 255)
check('course folder exists',     lambda: os.path.isdir(DRIVE))
check('subfolders created',       lambda: all(os.path.isdir(f'{DRIVE}/{d}')
                                              for d in ('data','outputs','notebooks')))
check('image written',            lambda: os.path.exists(PNG))
check('image round trip',         lambda: reloaded.shape == gray_u8.shape)
check('results round trip',       lambda: len(pd.read_csv(CSV)) == 5)

print()
for name, ok in checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}]  {name}')

score = sum(checks.values())
print(f'\n{score}/{len(checks)} passed')
print('You are ready for Lab 1.' if score == len(checks)
      else 'Bring the failures to the first ten minutes of Lab 1.')


---
## 7 · Submission dry run

Practise now, so the real deadline is not the first time you try.

1. `Runtime → Restart and run all` — confirm it completes cleanly.
2. `File → Download → Download .ipynb`.
3. Upload it to the course page.

### Checklist

- [ ] Working in **my own copy** (title shows my name, not the original)
- [ ] Self-test prints **8/8 PASS**
- [ ] `lab00_results.csv` is in `cv_course/outputs/`
- [ ] Notebook runs top to bottom after a restart
- [ ] Q2.1, Q2.2 and Q4.1 answered

---

### Four things worth remembering

1. **Print `shape`, `dtype`, `min`, `max` before anything else.** Most errors
   in this course are visible in those four numbers.
2. **`rgb2gray` gives floats in 0–1.** Convert before saving.
3. **Save to Drive, not `/content/`.** Files in `/content/` vanish.
4. **Reload whatever you save.** A save you did not verify did not happen.

**Next:** read Chapter 1, including the worked examples — they are short, and
they are Lecture 1 in slow motion.
